# Практика 0. PyTorch за час

Этот ноутбук проходится **самостоятельно**, до первой практики. На занятиях мы его не разбираем.

Если вы уверенно пишете обучающий цикл на PyTorch — просто пролистайте и убедитесь,
что всё знакомо. Если нет — пройдите целиком, это займёт около часа.

Понадобится ровно четыре вещи: тензоры, автоматическое дифференцирование, `nn.Module`
и обучающий цикл. Всё остальное в курсе строится поверх них.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

torch.manual_seed(0)
print("версия PyTorch:", torch.__version__)
print("видеокарта доступна:", torch.cuda.is_available())

## 1. Тензоры

Тензор — многомерный массив, почти как в numpy, но умеет считаться на видеокарте
и запоминать историю операций.

In [ ]:
x = torch.tensor([[1., 2.], [3., 4.]])
print(x, x.shape, x.dtype, sep="\n")

print("\nсложение:", (x + 10).tolist())
print("матричное умножение:", (x @ x).tolist())
print("сумма по строкам:", x.sum(dim=1).tolist())
print("изменение формы:", x.reshape(4).tolist())

Три операции, которые будут встречаться в курсе постоянно:

- **индексирование списком** — `x[idx]` собирает строки в нужном порядке;
- **index_add_** — складывает строки по индексам (так работает агрегация соседей);
- **broadcast** — умножение матрицы на вектор-столбец.

In [ ]:
h = torch.tensor([[1., 1.], [2., 2.], [3., 3.]])
idx = torch.tensor([0, 2, 2])

print("собрали строки по индексам:\n", h[idx])

out = torch.zeros(3, 2)
out.index_add_(0, idx, h[idx])
print("\nсложили по индексам (агрегация):\n", out)

weights = torch.tensor([0.5, 1.0, 2.0]).unsqueeze(-1)
print("\nвзвесили строки:\n", h * weights)

## 2. Автоматическое дифференцирование

Если тензор помечен `requires_grad=True`, PyTorch запоминает все операции с ним
и умеет посчитать производные.

In [ ]:
w = torch.tensor([2.0], requires_grad=True)
loss = (w - 5) ** 2          # минимум в точке w = 5
loss.backward()
print("значение функции:", loss.item())
print("производная в точке w=2:", w.grad.item(), "  (проверьте руками: 2*(w-5) = -6)")

In [ ]:
# TODO: сделайте десять шагов градиентного спуска и убедитесь, что w приближается к 5
w = torch.tensor([2.0], requires_grad=True)
lr = 0.1
for step in range(10):
    ...
print("итог:", w.item())

Три правила, из-за которых чаще всего ломается обучение:

1. градиенты **накапливаются**, поэтому перед каждым шагом их обнуляют (`opt.zero_grad()`);
2. изменение весов делают внутри `torch.no_grad()` — иначе шаг попадёт в граф вычислений;
3. `loss.backward()` можно вызвать один раз на граф.

## 3. Модель как `nn.Module`

In [ ]:
class SmallNet(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SmallNet(4, 16, 3)
print(model)
print("параметров:", sum(p.numel() for p in model.parameters()))
print("выход на случайном входе:", model(torch.randn(5, 4)).shape)

## 4. Полный обучающий цикл

Соберём всё вместе на классической задаче — ирисы. Ровно этот цикл вы увидите
во всех практиках курса, только вместо `model(x)` будет `model(x, edge_index)`.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X = (X - X.mean(0)) / X.std(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

X_train = torch.tensor(X_train, dtype=torch.float)
X_test = torch.tensor(X_test, dtype=torch.float)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)
print("обучающая выборка:", X_train.shape)

In [ ]:
# TODO: допишите обучающий цикл
model = SmallNet(4, 16, 3)
opt = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(100):
    model.train()
    # 1. обнулить градиенты
    # 2. посчитать выход и лосс (F.cross_entropy)
    # 3. backward и шаг оптимизатора
    ...

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(X_test).argmax(1) == y_test).float().mean()
        print(f"эпоха {epoch:3d}: loss {loss.item():.3f}, accuracy {acc.item():.3f}")

## 5. Что нужно помнить к первой практике

- `model.train()` и `model.eval()` переключают поведение dropout — забыть про это
  классическая ошибка;
- вычисления без градиентов оборачивают в `torch.no_grad()`;
- `.item()` и `float()` достают число из тензора, `.detach().cpu().numpy()` — массив numpy;
- перенос на видеокарту: `model.to(device)` и `tensor.to(device)`, оба должны быть на одном устройстве.

Если этот ноутбук прошёлся без затруднений — вы готовы к курсу.